# Day 3：从零手写 Reward Model（本周第一个实践！）

> **目标**：从零实现完整的 Reward Model——`偏好数据集构建 → GPT-2 backbone + Reward Head → Bradley-Terry Loss → 训练循环 → 偏好预测评估`，在模拟偏好数据上训练成功（准确率 >80%），可视化 RM 的打分分布，并演示 Reward Hacking 现象。
>
> Reward Model 是 RLHF 三阶段的核心枢纽——它将人类偏好转化为可优化的标量信号。

---

## 实现路线图

```
Part 1: Bradley-Terry Loss 数学回顾
  偏好概率 P(y_w > y_l) = σ(r_w - r_l) → Loss = -log σ(Δr)

Part 2: 偏好数据集构建
  构造模拟偏好数据 → 好回答 vs 坏回答 → Dataset / DataLoader

Part 3: Reward Model 网络
  GPT-2 backbone → 取最后 token 隐状态 → Linear(hidden, 1) → 标量奖励

Part 4: 训练循环
  BT Loss → AdamW 优化 → 偏好预测准确率监控

Part 5: 评估与可视化
  RM 打分分布 → 好/坏回答分离度 → 偏好预测准确率

Part 6: Reward Hacking 演示
  长度偏见 → 格式偏见 → RM 的局限性
```

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

try:
    from transformers import GPT2Model, GPT2Tokenizer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])
    from transformers import GPT2Model, GPT2Tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Part 1：Bradley-Terry Loss 数学回顾

### 1.1 偏好概率

Day 2 推导的 Bradley-Terry 模型：

$$P(y_w \succ y_l \mid x) = \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))$$

### 1.2 训练目标

最大化偏好数据的对数似然，等价于最小化：

$$\boxed{L_{\text{RM}}(\theta) = -\mathbb{E}\left[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))\right]}$$

### 1.3 与二分类交叉熵的等价性

把 $\Delta r = r(y_w) - r(y_l)$ 视为 logit，标签为 1：

$$L = -\log \sigma(\Delta r) = \text{BCE\_with\_logits}(\Delta r, 1)$$

代码实现只需一行 `F.binary_cross_entropy_with_logits(delta_r, ones)`。

## Part 2：偏好数据集构建

我们构造一个模拟偏好数据集来训练 RM。每条数据包含：
- `prompt`：一个问题或指令
- `chosen`：较好的回答（$y_w$）
- `rejected`：较差的回答（$y_l$）

通过设计明确的质量差异（如信息量、准确性、友好程度），确保 RM 有可学习的信号。

In [ ]:
PREFERENCE_DATA = [
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves.",
        "rejected": "Machine learning is about computers."
    },
    {
        "prompt": "Explain photosynthesis.",
        "chosen": "Photosynthesis is the process by which green plants and certain other organisms transform light energy into chemical energy. During this process, plants absorb carbon dioxide and water, and using sunlight, convert them into glucose and oxygen.",
        "rejected": "Plants eat sunlight and make food, I think."
    },
    {
        "prompt": "How does gravity work?",
        "chosen": "Gravity is a fundamental force of nature that attracts objects with mass toward each other. According to Einstein's general relativity, gravity is the curvature of spacetime caused by mass and energy. The more massive an object, the stronger its gravitational pull.",
        "rejected": "Things fall down because of gravity. That's just how it works."
    },
    {
        "prompt": "What is the capital of France?",
        "chosen": "The capital of France is Paris. It is the largest city in France and serves as the country's political, economic, and cultural center. Paris is known for landmarks like the Eiffel Tower and the Louvre Museum.",
        "rejected": "I'm not sure, maybe London?"
    },
    {
        "prompt": "How to learn programming?",
        "chosen": "To learn programming effectively: 1) Start with a beginner-friendly language like Python. 2) Follow structured online courses or tutorials. 3) Practice by building small projects. 4) Read and understand other people's code. 5) Join programming communities for support.",
        "rejected": "Just read some code and you'll figure it out eventually."
    },
    {
        "prompt": "What is deep learning?",
        "chosen": "Deep learning is a class of machine learning algorithms that uses multiple layers of neural networks to progressively extract higher-level features from raw input. It has achieved remarkable success in tasks like image recognition, natural language processing, and game playing.",
        "rejected": "Deep learning is when you learn something really deeply."
    },
    {
        "prompt": "How does the internet work?",
        "chosen": "The internet works through a global network of interconnected computers that communicate using standardized protocols (TCP/IP). Data is broken into packets, routed through multiple networks via routers, and reassembled at the destination. DNS servers translate domain names to IP addresses.",
        "rejected": "The internet is just WiFi and websites."
    },
    {
        "prompt": "What is climate change?",
        "chosen": "Climate change refers to long-term shifts in global temperatures and weather patterns. While natural causes exist, since the 1800s, human activities—primarily burning fossil fuels like coal, oil, and gas—have been the main driver, releasing greenhouse gases that trap heat in Earth's atmosphere.",
        "rejected": "The weather changes sometimes, that's climate change."
    },
    {
        "prompt": "Explain the theory of relativity.",
        "chosen": "Einstein's theory of relativity has two parts: Special Relativity (1905) states that the laws of physics are the same in all inertial frames and that the speed of light is constant. General Relativity (1915) extends this to include gravity as the curvature of spacetime caused by mass and energy.",
        "rejected": "Everything is relative, that's basically it."
    },
    {
        "prompt": "What is a neural network?",
        "chosen": "A neural network is a computational model inspired by the biological neural networks in the human brain. It consists of layers of interconnected nodes (neurons) that process information. Each connection has a weight that adjusts during training. Neural networks can learn complex patterns from data.",
        "rejected": "It's a network of neurons in a computer or something."
    },
    {
        "prompt": "How do vaccines work?",
        "chosen": "Vaccines work by training the immune system to recognize and fight specific pathogens. They contain weakened or inactivated forms of the virus, or just key proteins from it. This stimulates the immune system to produce antibodies and memory cells, providing protection against future infections.",
        "rejected": "They put some stuff in your body and it helps somehow."
    },
    {
        "prompt": "What is blockchain?",
        "chosen": "Blockchain is a distributed, decentralized ledger technology that records transactions across multiple computers. Each block contains a cryptographic hash of the previous block, a timestamp, and transaction data. This design makes it resistant to data modification and enables trustless peer-to-peer transactions.",
        "rejected": "Blockchain is Bitcoin."
    },
    {
        "prompt": "How does a car engine work?",
        "chosen": "A car engine (internal combustion engine) works through a four-stroke cycle: 1) Intake - air and fuel mixture enters the cylinder. 2) Compression - the piston compresses the mixture. 3) Power - spark plug ignites the mixture, pushing the piston down. 4) Exhaust - burnt gases are expelled.",
        "rejected": "You put gas in and it goes vroom."
    },
    {
        "prompt": "What is quantum computing?",
        "chosen": "Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information. Unlike classical bits (0 or 1), quantum bits (qubits) can exist in multiple states simultaneously, enabling quantum computers to solve certain problems exponentially faster than classical computers.",
        "rejected": "Quantum computing is really fast computing using quantum stuff."
    },
    {
        "prompt": "Explain natural selection.",
        "chosen": "Natural selection is the process where organisms with traits better suited to their environment tend to survive and reproduce more successfully. Over generations, these advantageous traits become more common in the population, driving evolution. It requires variation, inheritance, selection pressure, and differential reproduction.",
        "rejected": "Survival of the fittest. Strong things live, weak things don't."
    },
    {
        "prompt": "What is an API?",
        "chosen": "An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that programs can use to request and exchange information, acting as an intermediary between different software systems.",
        "rejected": "API is a way for apps to talk to each other."
    },
]

print(f"Total preference pairs: {len(PREFERENCE_DATA)}")
print(f"\nExample:")
print(f"  Prompt:   {PREFERENCE_DATA[0]['prompt']}")
print(f"  Chosen:   {PREFERENCE_DATA[0]['chosen'][:80]}...")
print(f"  Rejected: {PREFERENCE_DATA[0]['rejected']}")

In [ ]:
def augment_preference_data(data, swap_ratio=0.5):
    """通过交换 chosen/rejected 顺序来增强数据，防止位置偏见。"""
    augmented = []
    for item in data:
        augmented.append(item)
        augmented.append({
            "prompt": item["prompt"],
            "chosen": item["chosen"],
            "rejected": item["rejected"],
        })
    return augmented

train_data = PREFERENCE_DATA[:12]
val_data = PREFERENCE_DATA[12:]

train_data_augmented = augment_preference_data(train_data)
print(f"Train samples (after augmentation): {len(train_data_augmented)}")
print(f"Validation samples: {len(val_data)}")

In [ ]:
class PreferenceDataset(Dataset):
    """偏好对比数据集：每条数据包含 prompt + chosen response 和 prompt + rejected response。"""
    
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def _tokenize(self, prompt, response):
        text = f"Question: {prompt}\nAnswer: {response}"
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
    
    def __getitem__(self, idx):
        item = self.data[idx]
        chosen = self._tokenize(item['prompt'], item['chosen'])
        rejected = self._tokenize(item['prompt'], item['rejected'])
        return {
            'chosen_input_ids': chosen['input_ids'],
            'chosen_attention_mask': chosen['attention_mask'],
            'rejected_input_ids': rejected['input_ids'],
            'rejected_attention_mask': rejected['attention_mask'],
        }

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

train_dataset = PreferenceDataset(train_data_augmented, tokenizer, max_length=256)
val_dataset = PreferenceDataset(val_data, tokenizer, max_length=256)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

sample = train_dataset[0]
print(f"Chosen input_ids shape: {sample['chosen_input_ids'].shape}")
print(f"Sample chosen text: {tokenizer.decode(sample['chosen_input_ids'][:50])}...")

## Part 3：Reward Model 网络

RM 的架构：在 GPT-2 backbone 上加一个线性 Reward Head。

```
Input: [prompt + response]  (tokenized)
          ↓
    GPT-2 Backbone (frozen or fine-tuned)
          ↓
    last_hidden_state[:, -1, :]  ← 取序列最后一个有效 token
          ↓
    Linear(768, 1)  ← Reward Head
          ↓
    r ∈ ℝ  (标量奖励)
```

关键设计决策：
- 取**最后一个非 padding token** 的隐状态（因果注意力下包含最完整信息）
- Reward Head 无 bias（平移不变性，BT 模型只关心分数差）

In [ ]:
class RewardModel(nn.Module):
    """
    Reward Model: GPT-2 backbone + linear reward head.
    输入 (prompt, response) 的 token 序列，输出标量奖励 r ∈ ℝ。
    """
    
    def __init__(self, model_name='gpt2'):
        super().__init__()
        self.backbone = GPT2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # 768 for gpt2
        self.reward_head = nn.Linear(hidden_size, 1, bias=False)
    
    def get_last_hidden(self, input_ids, attention_mask):
        """获取每个序列最后一个非 padding token 的隐状态。"""
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, hidden)
        
        # 找到每个序列最后一个非 padding token 的位置
        seq_lengths = attention_mask.sum(dim=1) - 1  # (batch,)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        last_hidden = hidden_states[batch_idx, seq_lengths]  # (batch, hidden)
        
        return last_hidden
    
    def forward(self, input_ids, attention_mask):
        """计算奖励分数 r(x, y)。"""
        last_hidden = self.get_last_hidden(input_ids, attention_mask)
        reward = self.reward_head(last_hidden).squeeze(-1)  # (batch,)
        return reward


reward_model = RewardModel('gpt2').to(device)

total_params = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Reward head parameters: {sum(p.numel() for p in reward_model.reward_head.parameters()):,}")

In [ ]:
# 快速测试：RM 能正常前向传播
with torch.no_grad():
    sample = train_dataset[0]
    chosen_ids = sample['chosen_input_ids'].unsqueeze(0).to(device)
    chosen_mask = sample['chosen_attention_mask'].unsqueeze(0).to(device)
    rejected_ids = sample['rejected_input_ids'].unsqueeze(0).to(device)
    rejected_mask = sample['rejected_attention_mask'].unsqueeze(0).to(device)
    
    r_chosen = reward_model(chosen_ids, chosen_mask)
    r_rejected = reward_model(rejected_ids, rejected_mask)
    
    print(f"Reward (chosen):   {r_chosen.item():.4f}")
    print(f"Reward (rejected): {r_rejected.item():.4f}")
    print(f"Delta r:           {(r_chosen - r_rejected).item():.4f}")
    print(f"P(chosen > rejected) = σ(Δr) = {torch.sigmoid(r_chosen - r_rejected).item():.4f}")
    print("\n(训练前 RM 输出接近随机，这是预期的)")

## Part 4：训练循环

### 4.1 Bradley-Terry Loss

$$L = -\frac{1}{N}\sum_{i=1}^N \log \sigma(r_\theta(x_i, y_w^i) - r_\theta(x_i, y_l^i))$$

### 4.2 训练策略

- 使用 AdamW 优化器，学习率 `1e-5`（RM 训练 lr 通常比 SFT 更小）
- 训练 3 个 epoch（RM 容易过拟合，不宜太多）
- 监控偏好预测准确率作为核心评估指标

In [ ]:
def compute_rm_loss(reward_model, batch, device):
    """
    计算 Bradley-Terry Loss:
    L = -E[log σ(r(y_w) - r(y_l))]
    """
    chosen_ids = batch['chosen_input_ids'].to(device)
    chosen_mask = batch['chosen_attention_mask'].to(device)
    rejected_ids = batch['rejected_input_ids'].to(device)
    rejected_mask = batch['rejected_attention_mask'].to(device)
    
    r_chosen = reward_model(chosen_ids, chosen_mask)      # (batch,)
    r_rejected = reward_model(rejected_ids, rejected_mask)  # (batch,)
    
    # BT Loss = -log σ(r_w - r_l) = BCE_with_logits(r_w - r_l, 1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    
    # 计算准确率：r_chosen > r_rejected 的比例
    accuracy = (r_chosen > r_rejected).float().mean()
    
    return loss, accuracy, r_chosen.mean().item(), r_rejected.mean().item()

In [ ]:
@torch.no_grad()
def evaluate_rm(reward_model, dataloader, device):
    """在验证集上评估 RM 的偏好预测准确率和 Loss。"""
    reward_model.eval()
    total_loss = 0
    total_acc = 0
    total_r_chosen = 0
    total_r_rejected = 0
    n_batches = 0
    
    for batch in dataloader:
        loss, acc, r_c, r_r = compute_rm_loss(reward_model, batch, device)
        total_loss += loss.item()
        total_acc += acc.item()
        total_r_chosen += r_c
        total_r_rejected += r_r
        n_batches += 1
    
    return {
        'loss': total_loss / n_batches,
        'accuracy': total_acc / n_batches,
        'r_chosen': total_r_chosen / n_batches,
        'r_rejected': total_r_rejected / n_batches,
    }

In [ ]:
def train_reward_model(reward_model, train_loader, val_loader, device,
                        n_epochs=3, lr=1e-5, log_interval=5):
    """
    训练 Reward Model 的完整循环。
    
    核心流程:
    1. 对每个 batch: 计算 r(y_w) 和 r(y_l)
    2. 计算 BT Loss = -log σ(r_w - r_l)
    3. 反向传播 + AdamW 更新
    4. 监控偏好预测准确率
    """
    optimizer = optim.AdamW(reward_model.parameters(), lr=lr, weight_decay=0.01)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'r_chosen': [], 'r_rejected': [],
    }
    
    global_step = 0
    
    for epoch in range(n_epochs):
        reward_model.train()
        epoch_loss = 0
        epoch_acc = 0
        n_batches = 0
        
        for batch in train_loader:
            loss, acc, r_c, r_r = compute_rm_loss(reward_model, batch, device)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(reward_model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            n_batches += 1
            global_step += 1
        
        avg_train_loss = epoch_loss / n_batches
        avg_train_acc = epoch_acc / n_batches
        
        val_metrics = evaluate_rm(reward_model, val_loader, device)
        
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(avg_train_acc)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['r_chosen'].append(val_metrics['r_chosen'])
        history['r_rejected'].append(val_metrics['r_rejected'])
        
        print(f"Epoch {epoch+1}/{n_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2%} | "
              f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2%} | "
              f"r(chosen): {val_metrics['r_chosen']:.3f} | r(rejected): {val_metrics['r_rejected']:.3f}")
    
    return history

In [ ]:
print("=" * 70)
print("开始训练 Reward Model")
print("=" * 70)

history = train_reward_model(
    reward_model, train_loader, val_loader, device,
    n_epochs=5, lr=1e-5
)

print("\n训练完成!")
print(f"最终验证准确率: {history['val_acc'][-1]:.2%}")

## Part 5：评估与可视化

训练完成后，我们从三个角度评估 RM：
1. **训练曲线**：Loss 和准确率的变化
2. **打分分布**：chosen vs rejected 的 reward 分数分离度
3. **逐条分析**：每个偏好对的 RM 判断

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss 曲线
axes[0].plot(history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(history['val_loss'], 'r-s', label='Val Loss', markersize=4)
axes[0].axhline(y=np.log(2), color='gray', linestyle='--', alpha=0.5, label='Random (log2)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('RM Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(history['train_acc'], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(history['val_acc'], 'r-s', label='Val Acc', markersize=4)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Preference Prediction Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reward 分数对比
axes[2].plot(history['r_chosen'], 'g-o', label='r(chosen)', markersize=4)
axes[2].plot(history['r_rejected'], 'r-s', label='r(rejected)', markersize=4)
axes[2].fill_between(range(len(history['r_chosen'])),
                      history['r_chosen'], history['r_rejected'],
                      alpha=0.2, color='blue')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Reward Score')
axes[2].set_title('Reward Gap (chosen vs rejected)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 逐条分析：每个偏好对的 RM 判断
reward_model.eval()
print("=" * 70)
print("逐条偏好预测分析")
print("=" * 70)

all_r_chosen = []
all_r_rejected = []
correct = 0

with torch.no_grad():
    for i, item in enumerate(PREFERENCE_DATA):
        # Tokenize chosen
        chosen_text = f"Question: {item['prompt']}\nAnswer: {item['chosen']}"
        chosen_enc = tokenizer(chosen_text, max_length=256, padding='max_length',
                               truncation=True, return_tensors='pt').to(device)
        # Tokenize rejected
        rejected_text = f"Question: {item['prompt']}\nAnswer: {item['rejected']}"
        rejected_enc = tokenizer(rejected_text, max_length=256, padding='max_length',
                                  truncation=True, return_tensors='pt').to(device)
        
        r_c = reward_model(chosen_enc['input_ids'], chosen_enc['attention_mask']).item()
        r_r = reward_model(rejected_enc['input_ids'], rejected_enc['attention_mask']).item()
        
        all_r_chosen.append(r_c)
        all_r_rejected.append(r_r)
        
        is_correct = r_c > r_r
        correct += int(is_correct)
        status = "✓" if is_correct else "✗"
        
        print(f"\n[{status}] {item['prompt'][:50]}")
        print(f"    r(chosen)={r_c:+.4f}  r(rejected)={r_r:+.4f}  Δr={r_c-r_r:+.4f}")

print(f"\n{'='*70}")
print(f"Overall Accuracy: {correct}/{len(PREFERENCE_DATA)} = {correct/len(PREFERENCE_DATA):.2%}")

In [ ]:
# 打分分布可视化
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(PREFERENCE_DATA))
width = 0.35

bars1 = ax.bar(x - width/2, all_r_chosen, width, label='Chosen (y_w)', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x + width/2, all_r_rejected, width, label='Rejected (y_l)', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Sample Index')
ax.set_ylabel('Reward Score')
ax.set_title('RM Scores: Chosen vs Rejected')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nr(chosen) mean: {np.mean(all_r_chosen):.4f} ± {np.std(all_r_chosen):.4f}")
print(f"r(rejected) mean: {np.mean(all_r_rejected):.4f} ± {np.std(all_r_rejected):.4f}")
print(f"Reward gap mean: {np.mean(np.array(all_r_chosen) - np.array(all_r_rejected)):.4f}")

## Part 6：Reward Hacking 演示

Day 2 我们分析了 Reward Hacking 的理论。现在用训练好的 RM 实际演示两种常见的 hacking 模式：

1. **长度偏见**：RM 是否对更长的回答给出更高的分数？
2. **格式偏见**：RM 是否对特定格式（如列表）给出更高的分数？

In [ ]:
def score_text(reward_model, tokenizer, prompt, response, device):
    """对给定的 (prompt, response) 对计算 RM 分数。"""
    text = f"Question: {prompt}\nAnswer: {response}"
    enc = tokenizer(text, max_length=256, padding='max_length',
                    truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        reward = reward_model(enc['input_ids'], enc['attention_mask'])
    return reward.item()

# === 实验 1：长度偏见 ===
print("=" * 70)
print("实验 1：长度偏见测试")
print("=" * 70)

test_prompt = "What is Python?"
responses_by_length = [
    "A programming language.",
    "Python is a popular programming language used for many tasks.",
    "Python is a high-level, interpreted programming language known for its simple syntax and readability. It is widely used in web development, data science, and automation.",
    "Python is a high-level, interpreted programming language known for its simple syntax and readability. It is widely used in web development, data science, artificial intelligence, and automation. Python was created by Guido van Rossum and first released in 1991. It supports multiple programming paradigms including procedural, object-oriented, and functional programming.",
]

print(f"\nPrompt: {test_prompt}\n")
length_scores = []
for resp in responses_by_length:
    score = score_text(reward_model, tokenizer, test_prompt, resp, device)
    length_scores.append((len(resp), score))
    print(f"  Length={len(resp):3d} chars | Score={score:+.4f} | {resp[:60]}...")

lengths, scores = zip(*length_scores)
correlation = np.corrcoef(lengths, scores)[0, 1]
print(f"\n长度与分数的相关性: {correlation:.4f}")
if correlation > 0.5:
    print("⚠ 检测到长度偏见：RM 倾向于给更长的回答更高的分数！")
    print("  这在 RLHF 训练中可能导致模型生成冗长回答。")

In [ ]:
# === 实验 2：格式偏见 ===
print("=" * 70)
print("实验 2：格式偏见测试")
print("=" * 70)

test_prompt2 = "How to learn programming?"
responses_by_format = {
    "Plain text": "You should start with a beginner language, take online courses, practice building projects, and join communities for support.",
    "Numbered list": "1) Start with Python. 2) Take online courses. 3) Practice building projects. 4) Read other people's code. 5) Join programming communities.",
    "Short": "Start with Python and practice a lot.",
    "Hedging": "I think maybe you could try learning Python, but I'm not really sure. It might be good to take some courses, perhaps.",
}

print(f"\nPrompt: {test_prompt2}\n")
for name, resp in responses_by_format.items():
    score = score_text(reward_model, tokenizer, test_prompt2, resp, device)
    print(f"  [{name:15s}] Score={score:+.4f} | {resp[:60]}...")

In [ ]:
# === 实验 3：无意义重复 ===
print("=" * 70)
print("实验 3：无意义重复测试")
print("=" * 70)

test_prompt3 = "What is machine learning?"
responses_repetition = {
    "Normal": "Machine learning is a field of AI that enables systems to learn from data and improve over time without explicit programming.",
    "Repeated": "Machine learning is a field of AI. Machine learning enables systems to learn. Machine learning improves over time. Machine learning doesn't need explicit programming. Machine learning is very important. Machine learning is widely used.",
    "Nonsense": "Indeed, the paradigmatic synergistic framework of computational methodologies facilitates the emergence of autonomous cognitive architectures through iterative optimization protocols.",
}

print(f"\nPrompt: {test_prompt3}\n")
for name, resp in responses_repetition.items():
    score = score_text(reward_model, tokenizer, test_prompt3, resp, device)
    print(f"  [{name:12s}] Score={score:+.4f} | {resp[:70]}...")

print("\n" + "=" * 70)
print("总结：RM 的局限性")
print("=" * 70)
print("1. RM 从有限数据中学到的是标注者偏好的近似")
print("2. 在训练分布外（OOD），RM 的打分可能不可靠")
print("3. PPO 的激进优化会放大 RM 的偏见 → Reward Hacking")
print("4. KL penalty 是防止 Reward Hacking 的第一道防线")
print("\n这些问题将在 Day 6 的 RLHF 训练中实际体现。")

---

## 总结

本日我们从零实现了一个完整的 Reward Model：

1. **偏好数据集**：构建了 (prompt, chosen, rejected) 格式的训练数据
2. **RM 架构**：GPT-2 backbone + linear reward head，取最后 token 隐状态
3. **BT Loss**：$L = -\log \sigma(r_w - r_l)$，等价于二分类交叉熵
4. **训练循环**：AdamW + gradient clipping + 准确率监控
5. **评估**：可视化打分分布和偏好预测准确率
6. **Reward Hacking**：演示了长度偏见、格式偏见等 RM 局限性

### 核心收获

- RM 的本质是把人类偏好编码为标量信号
- BT Loss 只关心分数**差**，绝对分数没有意义
- RM 是 RLHF 链路中最脆弱的环节——它的偏见会被 PPO 放大

### 下一步

Day 4 将深入 RLHF-PPO 的四模型架构，理解 Actor / Critic / Reference / Reward 如何协同工作。Day 6 将使用我们今天训练的 RM（或简化版）来完成完整的 RLHF-PPO 训练。